In [ ]:
!git clone --depth=1 https://github.com/pandas-dev/pandas.git

In [ ]:
!pip install pandas tqdm

In [ ]:
import ast
import json
import logging
from pathlib import Path
from typing import Iterator, Dict, Optional, Any, Union
from tqdm import tqdm

# Configure production-grade logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

class CodeExtractor(ast.NodeVisitor):
    """
    A custom AST visitor that tracks class hierarchy to extract context-rich
    method names (e.g., 'ClassName.method_name') alongside standalone functions.
    """
    def __init__(self, source_code: str):
        self.source_code = source_code
        self.pairs: list[Dict[str, str]] = []
        self.current_class: Optional[str] = None

    def visit_ClassDef(self, node: ast.ClassDef) -> None:
        # Track the class context before diving into its methods
        prev_class = self.current_class
        self.current_class = node.name
        self.generic_visit(node)
        # Restore context after exiting the class
        self.current_class = prev_class

    def visit_FunctionDef(self, node: ast.FunctionDef) -> None:
        self._process_function(node)
        self.generic_visit(node)

    def visit_AsyncFunctionDef(self, node: ast.AsyncFunctionDef) -> None:
        self._process_function(node)
        self.generic_visit(node)

    def _process_function(self, node: Union[ast.FunctionDef, ast.AsyncFunctionDef]) -> None:
        name = getattr(node, "name", "")

        # Skip private/internal methods, but keep __init__
        if name.startswith("_") and name != "__init__":
            return

        # ast.get_docstring handles dedenting and basic cleanup
        docstring = ast.get_docstring(node, clean=True)
        if not docstring:
            return

        # Extract only the primary summary (first logical block)
        clean_docstring = docstring.split("\n\n")[0].strip()

        # Stricter filter: Ignore trivial or overly short docstrings
        if len(clean_docstring) < 15:
            return

        try:
            # Extract the raw source code
            code_snippet = ast.get_source_segment(self.source_code, node)
            if not code_snippet:
                return

            # Prepend class name if the function is a method
            full_name = f"{self.current_class}.{name}" if self.current_class else name

            self.pairs.append({
                "function_name": full_name,
                "docstring": clean_docstring,
                "code": code_snippet.strip()
            })
        except Exception as e:
            # Catch segment extraction failures without breaking the whole file
            logger.debug(f"Failed to extract source for {name}: {e}")


def process_file(file_path: Path) -> Iterator[Dict[str, str]]:
    """Reads a file and yields extracted code pairs efficiently."""
    try:
        # errors="replace" ensures strict UTF-8 doesn't crash on bad bytes
        source = file_path.read_text(encoding="utf-8", errors="replace")
        tree = ast.parse(source)

        extractor = CodeExtractor(source)
        extractor.visit(tree)

        yield from extractor.pairs

    except SyntaxError:
        logger.debug(f"Syntax error skipped in: {file_path}")
    except Exception as e:
        logger.warning(f"Unexpected error processing {file_path}: {e}")


def main(repo_dir: str, output_file: str) -> None:
    repo_path = Path(repo_dir)
    out_path = Path(output_file)

    if not repo_path.exists() or not repo_path.is_dir():
        logger.error(f"Repository directory '{repo_dir}' not found.")
        return

    exclude_dirs = {"tests", "benchmarks", "docs", ".git", "build", "scripts"}

    logger.info(f"Scanning {repo_dir} for Python files...")

    # Efficient path traversal filtering out unwanted directories
    python_files = [
        p for p in repo_path.rglob("*.py")
        if not any(part in exclude_dirs for part in p.parts)
    ]

    logger.info(f"Found {len(python_files)} target Python files. Starting extraction...")

    # Ensure output directory exists
    out_path.parent.mkdir(parents=True, exist_ok=True)

    total_extracted = 0

    # Stream writes directly to disk to minimize RAM usage
    with out_path.open("w", encoding="utf-8") as f:
        for file_path in tqdm(python_files, desc="Parsing ASTs"):
            for pair in process_file(file_path):
                f.write(json.dumps(pair) + "\n")
                total_extracted += 1

    logger.info(f"Successfully extracted {total_extracted} code-docstring pairs to {output_file}")


if __name__ == "__main__":
    # Point this to your cloned pandas directory
    main(repo_dir="./pandas", output_file="pandas_dataset.jsonl")

In [ ]:
!pip install sentence-transformers torch

In [5]:
import json
import time
import torch
import logging
from pathlib import Path
from typing import List, Dict, Any
from sentence_transformers import SentenceTransformer, util

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

def get_optimal_device() -> str:
    """Detects the best available hardware accelerator."""
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    return "cpu"

def load_corpus(file_path: Path) -> List[Dict[str, str]]:
    """Loads the extracted code-docstring pairs efficiently."""
    if not file_path.exists():
        raise FileNotFoundError(f"Corpus file not found: {file_path}")

    corpus = []
    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                corpus.append(json.loads(line))
    return corpus

def get_or_create_embeddings(model: SentenceTransformer, corpus_code: List[str], cache_path: Path) -> torch.Tensor:
    """Loads embeddings from disk if available, otherwise generates and caches them."""
    if cache_path.exists():
        logger.info(f"Loading cached embeddings from {cache_path}...")
        return torch.load(cache_path, weights_only=True)

    logger.info(f"Cache miss. Embedding {len(corpus_code)} snippets (this might take a minute)...")
    start_time = time.time()

    # Increase batch size for better GPU utilization if not on CPU
    batch_size = 32 if model.device.type == "cpu" else 128

    corpus_embeddings = model.encode(
        corpus_code,
        convert_to_tensor=True,
        show_progress_bar=True,
        batch_size=batch_size
    )

    logger.info(f"Embedding complete in {time.time() - start_time:.2f} seconds.")

    # Cache the embeddings for future runs
    torch.save(corpus_embeddings, cache_path)
    logger.info(f"Embeddings cached to {cache_path}")

    return corpus_embeddings

def main():
    corpus_path = Path("pandas_dataset.jsonl")
    cache_path = Path("baseline_embeddings.pt")
    output_report_path = Path("baseline_eval_results.json")

    logger.info("Loading dataset...")
    corpus = load_corpus(corpus_path)
    corpus_code = [item["code"] for item in corpus]

    device = get_optimal_device()
    logger.info(f"Loading baseline model: all-MiniLM-L6-v2 on [{device.upper()}]...")
    model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

    corpus_embeddings = get_or_create_embeddings(model, corpus_code, cache_path)

    # The Evaluation Set
    queries = [
        "parse a date column from a string",
        "drop rows with missing values",
        "group data by a column and calculate the mean",
        "merge two dataframes together on a specific key",
        "convert dataframe to a dictionary"
    ]

    logger.info("=== RUNNING BASELINE EVALUATION ===")

    eval_report: List[Dict[str, Any]] = []

    for query in queries:
        print(f"\nQuery: '{query}'")

        query_embedding = model.encode(query, convert_to_tensor=True)
        hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=3)[0]

        query_results = {
            "query": query,
            "top_hits": []
        }

        print("Top 3 Results:")
        for i, hit in enumerate(hits):
            idx = int(hit["corpus_id"])
            score = hit["score"]
            func_name = corpus[idx]["function_name"]

            # Save to report
            query_results["top_hits"].append({
                "rank": i + 1,
                "function_name": func_name,
                "score": float(score) # Convert tensor float to standard python float for JSON
            })

            print(f"  {i+1}. {func_name} (Score: {score:.4f})")

        eval_report.append(query_results)
        print("-" * 50)

    # Export the baseline report
    with output_report_path.open("w", encoding="utf-8") as f:
        json.dump(eval_report, f, indent=4)

    logger.info(f"Baseline evaluation saved to {output_report_path}. Keep this file for Phase 4 comparison.")

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]


Query: 'parse a date column from a string'
Top 3 Results:
  1. validate_parse_dates_presence (Score: 0.5009)
  2. ExcelWriter.date_format (Score: 0.4944)
  3. ExcelWriter.datetime_format (Score: 0.4692)
--------------------------------------------------

Query: 'drop rows with missing values'
Top 3 Results:
  1. DataFrame.dropna (Score: 0.6381)
  2. Series.dropna (Score: 0.5781)
  3. ExtensionArray.dropna (Score: 0.4801)
--------------------------------------------------

Query: 'group data by a column and calculate the mean'
Top 3 Results:
  1. Resampler.mean (Score: 0.4331)
  2. GroupBy.ohlc (Score: 0.4073)
  3. Resampler.nunique (Score: 0.4039)
--------------------------------------------------

Query: 'merge two dataframes together on a specific key'
Top 3 Results:
  1. DataFrame.combine (Score: 0.6164)
  2. DataFrame.combine_first (Score: 0.6070)
  3. merge_asof (Score: 0.6057)
--------------------------------------------------

Query: 'convert dataframe to a dictionary'
Top 3 Re

In [6]:
!pip install rank_bm25

In [7]:
import json
import re
import random
import logging
import heapq
from pathlib import Path
from tqdm import tqdm
from rank_bm25 import BM25Okapi

# Configure production-grade logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

def tokenize(text: str) -> list[str]:
    """
    Advanced Python code tokenizer.
    Splits on non-alphanumeric characters AND breaks apart camelCase.
    """
    # Insert a space before capital letters to break camelCase
    text = re.sub(r'(?<!^)(?=[A-Z])', ' ', text)
    # Split by non-alphanumeric
    tokens = re.split(r'[^a-zA-Z0-9]+', text)
    return [t.lower() for t in tokens if len(t) > 1]

def load_corpus(file_path: Path) -> list[dict]:
    if not file_path.exists():
        raise FileNotFoundError(f"Corpus file not found: {file_path}")

    corpus = []
    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                corpus.append(json.loads(line))
    return corpus

def main():
    input_path = Path("pandas_dataset.jsonl")
    output_path = Path("training_triplets.jsonl")

    logger.info("Loading dataset...")
    corpus = load_corpus(input_path)

    logger.info("Tokenizing code corpus for BM25...")
    tokenized_corpus = [tokenize(item["code"]) for item in tqdm(corpus, desc="Tokenizing")]

    logger.info("Building BM25 Index (this takes a moment)...")
    bm25 = BM25Okapi(tokenized_corpus)

    logger.info("Mining hard negatives for each query...")

    # Open file immediately for streaming writes
    with output_path.open("w", encoding="utf-8") as f:
        for i, item in enumerate(tqdm(corpus, desc="Mining Triplets")):
            anchor_query = item["docstring"]
            positive_code = item["code"]

            # Tokenize the positive code to find structurally similar code
            tokenized_query = tokenize(positive_code)

            # Get BM25 scores for all snippets
            scores = bm25.get_scores(tokenized_query)

            # OPTIMIZATION: Use heapq to efficiently find the top 10 indices without sorting the whole array
            top_10_indices = heapq.nlargest(10, range(len(scores)), key=scores.__getitem__)

            hard_negative_code = None

            for idx in top_10_indices:
                if idx != i: # Skip the true positive
                    candidate_code = corpus[idx]["code"]
                    candidate_docstring = corpus[idx]["docstring"]

                    # FALSE NEGATIVE TRAP: Ensure we don't pick an exact code match OR an alias (identical docstring)
                    if candidate_code != positive_code and candidate_docstring != anchor_query:
                        hard_negative_code = candidate_code
                        break

            # Fallback to a random snippet if BM25 fails to find a valid negative
            if not hard_negative_code:
                valid_random_indices = [x for x in range(len(corpus)) if x != i]
                random_idx = random.choice(valid_random_indices)
                hard_negative_code = corpus[random_idx]["code"]

            # Construct triplet and stream directly to disk
            triplet = {
                "query": anchor_query,
                "positive": positive_code,
                "negative": hard_negative_code
            }
            f.write(json.dumps(triplet) + "\n")

    logger.info(f"Phase 3 Complete: Mined {len(corpus)} triplets successfully to {output_path}!")

if __name__ == "__main__":
    main()

Mining Triplets: 100%|██████████| 2266/2266 [04:23<00:00,  8.60it/s]


In [9]:
import json
import logging
import random
import torch
import math
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import TripletEvaluator

# Configure production-grade logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

def get_optimal_device() -> str:
    """Detects the best available hardware accelerator."""
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    return "cpu"

def load_triplets(file_path: Path) -> list[InputExample]:
    """Loads the JSONL triplets and converts them into SentenceTransformer InputExamples."""
    if not file_path.exists():
        raise FileNotFoundError(f"Triplets file not found: {file_path}")

    examples = []
    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                examples.append(InputExample(
                    texts=[data["query"], data["positive"], data["negative"]]
                ))
    return examples

def main():
    triplets_path = Path("training_triplets.jsonl")
    model_save_path = "finetuned-pandas-coder"

    device = get_optimal_device()
    logger.info(f"Hardware detected: [{device.upper()}]. Loading baseline model...")

    model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

    logger.info(f"Loading training triplets from {triplets_path}...")
    all_examples = load_triplets(triplets_path)

    # 1. Train/Validation Split (90/10)
    random.seed(42) # Set seed for reproducibility
    random.shuffle(all_examples)

    split_idx = int(len(all_examples) * 0.9)
    train_examples = all_examples[:split_idx]
    val_examples = all_examples[split_idx:]

    logger.info(f"Dataset split: {len(train_examples)} Training | {len(val_examples)} Validation")

    # 2. Dynamic Batch Sizing
    # MNRL benefits massively from larger batch sizes.
    batch_size = 32 if device in ["cuda", "mps"] else 16
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

    # 3. Setup the Evaluator
    logger.info("Configuring Triplet Evaluator...")
    val_anchors = [ex.texts[0] for ex in val_examples]
    val_positives = [ex.texts[1] for ex in val_examples]
    val_negatives = [ex.texts[2] for ex in val_examples]

    evaluator = TripletEvaluator(
        anchors=val_anchors,
        positives=val_positives,
        negatives=val_negatives,
        name="pandas-val",
        show_progress_bar=False
    )

    logger.info("Initializing Multiple Negatives Ranking Loss...")
    train_loss = losses.MultipleNegativesRankingLoss(model=model)

    # Calculate evaluation steps (evaluate roughly 3 times per epoch)
    eval_steps = math.ceil(len(train_dataloader) / 3)

    logger.info("=== STARTING TRAINING LOOP ===")
    logger.info(f"Epochs: 4 | Batch Size: {batch_size} | Eval Steps: {eval_steps}")

    # 4. Fine-tune with explicit hyperparameter control and checkpointing
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        evaluator=evaluator,
        epochs=4,
        evaluation_steps=eval_steps,
        warmup_steps=100,
        output_path=model_save_path,
        save_best_model=True, # Critical: Only saves the weights that score highest on the evaluator
        show_progress_bar=True,
        optimizer_params={'lr': 2e-5} # Standard safe learning rate for embedding fine-tuning
    )

    logger.info(f"Training complete! Best model successfully written to ./{model_save_path}")

if __name__ == "__main__":
    main()

/tmp/ipykernel_1052/1104531678.py:8: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses
/tmp/ipykernel_1052/1104531678.py:9: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import TripletEvaluator


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Pandas-val Cosine Accuracy
22,No log,No log,0.942731
44,No log,No log,0.969163
64,No log,No log,0.982379
66,No log,No log,0.982379
88,No log,No log,0.986784
110,No log,No log,0.995595
128,No log,No log,0.991189
132,No log,No log,0.995595
154,No log,No log,1.000000
176,No log,No log,0.995595


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

**Testing with new model: Using our evaluate_baseline.py code and pointing to our new model**

In [10]:
import json
import time
import torch
import logging
from pathlib import Path
from typing import List, Dict, Any
from sentence_transformers import SentenceTransformer, util

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

def get_optimal_device() -> str:
    """Detects the best available hardware accelerator."""
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    return "cpu"

def load_corpus(file_path: Path) -> List[Dict[str, str]]:
    """Loads the extracted code-docstring pairs efficiently."""
    if not file_path.exists():
        raise FileNotFoundError(f"Corpus file not found: {file_path}")

    corpus = []
    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                corpus.append(json.loads(line))
    return corpus

def get_or_create_embeddings(model: SentenceTransformer, corpus_code: List[str], cache_path: Path) -> torch.Tensor:
    """Loads embeddings from disk if available, otherwise generates and caches them."""
    if cache_path.exists():
        logger.info(f"Loading cached embeddings from {cache_path}...")
        return torch.load(cache_path, weights_only=True)

    logger.info(f"Cache miss. Embedding {len(corpus_code)} snippets (this might take a minute)...")
    start_time = time.time()

    # Increase batch size for better GPU utilization if not on CPU
    batch_size = 32 if model.device.type == "cpu" else 128

    corpus_embeddings = model.encode(
        corpus_code,
        convert_to_tensor=True,
        show_progress_bar=True,
        batch_size=batch_size
    )

    logger.info(f"Embedding complete in {time.time() - start_time:.2f} seconds.")

    # Cache the embeddings for future runs
    torch.save(corpus_embeddings, cache_path)
    logger.info(f"Embeddings cached to {cache_path}")

    return corpus_embeddings

def main():
    corpus_path = Path("pandas_dataset.jsonl")
    cache_path = Path("finetuned_embeddings.pt")
    output_report_path = Path("finetuned_eval_results.json")

    logger.info("Loading dataset...")
    corpus = load_corpus(corpus_path)
    corpus_code = [item["code"] for item in corpus]

    device = get_optimal_device()
    logger.info(f"Loading FINETUNED model on [{device.upper()}]...")
    # Point directly to the folder where your trained weights were saved
    model = SentenceTransformer("./finetuned-pandas-coder", device=device)

    corpus_embeddings = get_or_create_embeddings(model, corpus_code, cache_path)

    # The Evaluation Set
    queries = [
        "parse a date column from a string",
        "drop rows with missing values",
        "group data by a column and calculate the mean",
        "merge two dataframes together on a specific key",
        "convert dataframe to a dictionary"
    ]

    logger.info("=== RUNNING BASELINE EVALUATION ===")

    eval_report: List[Dict[str, Any]] = []

    for query in queries:
        print(f"\nQuery: '{query}'")

        query_embedding = model.encode(query, convert_to_tensor=True)
        hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=3)[0]

        query_results = {
            "query": query,
            "top_hits": []
        }

        print("Top 3 Results:")
        for i, hit in enumerate(hits):
            idx = int(hit["corpus_id"])
            score = hit["score"]
            func_name = corpus[idx]["function_name"]

            # Save to report
            query_results["top_hits"].append({
                "rank": i + 1,
                "function_name": func_name,
                "score": float(score) # Convert tensor float to standard python float for JSON
            })

            print(f"  {i+1}. {func_name} (Score: {score:.4f})")

        eval_report.append(query_results)
        print("-" * 50)

    # Export the baseline report
    with output_report_path.open("w", encoding="utf-8") as f:
        json.dump(eval_report, f, indent=4)

    logger.info(f"Baseline evaluation saved to {output_report_path}. Keep this file for Phase 4 comparison.")

if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]


Query: 'parse a date column from a string'
Top 3 Results:
  1. validate_parse_dates_presence (Score: 0.4582)
  2. parse_datetime_format_str (Score: 0.4131)
  3. ExcelWriter.date_format (Score: 0.3780)
--------------------------------------------------

Query: 'drop rows with missing values'
Top 3 Results:
  1. DataFrame.dropna (Score: 0.6842)
  2. Categorical.isna (Score: 0.5427)
  3. ExtensionArray.isna (Score: 0.5338)
--------------------------------------------------

Query: 'group data by a column and calculate the mean'
Top 3 Results:
  1. GroupBy.mean (Score: 0.5163)
  2. Resampler.mean (Score: 0.4837)
  3. DataFrame.mean (Score: 0.4577)
--------------------------------------------------

Query: 'merge two dataframes together on a specific key'
Top 3 Results:
  1. DataFrame.combine (Score: 0.5253)
  2. merge_asof (Score: 0.5229)
  3. _MergeOperation.get_result (Score: 0.4589)
--------------------------------------------------

Query: 'convert dataframe to a dictionary'
Top 3 Res

In [11]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 104.5 MB/s eta 0:00:00


In [13]:
import json
import faiss
import torch
import numpy as np
import logging
import gc
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

# Configure production-grade logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

def get_optimal_device() -> str:
    if torch.cuda.is_available(): return "cuda"
    if torch.backends.mps.is_available(): return "mps"
    return "cpu"

def load_corpus(file_path: Path) -> list[dict]:
    if not file_path.exists():
        raise FileNotFoundError(f"Corpus file not found: {file_path}")
    corpus = []
    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                corpus.append(json.loads(line))
    return corpus

def main():
    model_path = "./finetuned-pandas-coder"
    corpus_path = Path("pandas_dataset.jsonl")
    index_save_path = "pandas_binary.faiss"
    metadata_save_path = "corpus_metadata.json"

    CHUNK_SIZE = 10000

    device = get_optimal_device()
    logger.info(f"Loading finetuned model on [{device.upper()}]...")
    model = SentenceTransformer(model_path, device=device)

    logger.info("Loading corpus...")
    corpus = load_corpus(corpus_path)
    corpus_code = [item["code"] for item in corpus]

    # Use the updated method name
    dimension = model.get_embedding_dimension()

    logger.info(f"Initializing FAISS Binary Index (Dimension: {dimension} bits)... dust")
    index = faiss.IndexBinaryFlat(dimension)

    total_float_bytes = 0
    total_binary_bytes = 0

    logger.info(f"Processing {len(corpus_code)} snippets in chunks of {CHUNK_SIZE}...")

    for i in tqdm(range(0, len(corpus_code), CHUNK_SIZE), desc="Indexing Batches"):
        batch_code = corpus_code[i : i + CHUNK_SIZE]

        float_embeddings = model.encode(
            batch_code,
            convert_to_tensor=True,
            show_progress_bar=False,
            batch_size=32 if device != "cpu" else 16
        )

        total_float_bytes += float_embeddings.element_size() * float_embeddings.nelement()

        # Use the updated quantization utility path
        binary_embeddings = util.quantize_embeddings(float_embeddings, precision="ubinary")
        total_binary_bytes += binary_embeddings.nbytes

        # Fix: binary_embeddings is already a numpy array
        binary_numpy = binary_embeddings.astype(np.uint8)

        index.add(binary_numpy)

        del float_embeddings
        del binary_embeddings
        del binary_numpy
        if device == "cuda":
            torch.cuda.empty_cache()
        gc.collect()

    float_size_mb = total_float_bytes / (1024 * 1024)
    binary_size_mb = total_binary_bytes / (1024 * 1024)
    logger.info("--- METRICS ---")
    logger.info(f"Total Vectors Indexed:    {index.ntotal}")
    logger.info(f"Float32 Memory Footprint: {float_size_mb:.4f} MB")
    logger.info(f"Binary Memory Footprint:  {binary_size_mb:.4f} MB")
    logger.info(f"Compression Ratio:        {float_size_mb / binary_size_mb:.1f}x smaller")
    logger.info("---------------")

    faiss.write_index_binary(index, index_save_path)
    logger.info(f"Binary index safely written to {index_save_path}")

    logger.info(f"Exporting metadata mapping to {metadata_save_path}...")
    with open(metadata_save_path, "w", encoding="utf-8") as f:
        compact_metadata = [
            {"id": i, "function_name": item["function_name"], "code": item["code"]}
            for i, item in enumerate(corpus)
        ]
        json.dump(compact_metadata, f)

    logger.info("Phase 5 Complete. Ready for deployment.")

if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexing Batches: 100%|██████████| 1/1 [00:04<00:00,  4.88s/it]


Verifying the Output of the Above Cell as Colab surpasses the Logger outputs

In [14]:
import faiss
import json
import os

print("=== PHASE 5 VERIFICATION ===")

# 1. Check physical file sizes on disk
try:
    faiss_size = os.path.getsize("pandas_binary.faiss") / 1024
    meta_size = os.path.getsize("corpus_metadata.json") / 1024
    print(f"FAISS Index Size:  {faiss_size:.2f} KB")
    print(f"Metadata Size:     {meta_size:.2f} KB")
except FileNotFoundError as e:
    print(f"ERROR: Missing file - {e}")

print("-" * 30)

# 2. Crack open the FAISS index
try:
    index = faiss.read_index_binary("pandas_binary.faiss")
    print(f"Total vectors locked in FAISS: {index.ntotal}")
    print(f"Index dimensionality (bits):   {index.d}")
except Exception as e:
    print(f"FAISS Load Error: {e}")

# 3. Crack open the Metadata
try:
    with open("corpus_metadata.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)
    print(f"Total entries in metadata:     {len(metadata)}")
except Exception as e:
    print(f"Metadata Load Error: {e}")

print("-" * 30)

# 4. The Final Check
if 'index' in locals() and 'metadata' in locals():
    if index.ntotal == len(metadata):
        print("✅ SUCCESS: FAISS index and metadata are perfectly perfectly synced! You are ready for Phase 6.")
    else:
        print("❌ MISMATCH: Your vector count does not match your metadata count.")

=== PHASE 5 VERIFICATION ===
FAISS Index Size:  106.25 KB
Metadata Size:     4610.37 KB
------------------------------
Total vectors locked in FAISS: 2266
Index dimensionality (bits):   384
Total entries in metadata:     2266
------------------------------
✅ SUCCESS: FAISS index and metadata are perfectly perfectly synced! You are ready for Phase 6.
